# M2 – So sánh baseline K-Means và DBSCAN (mục 5.3)

Phụ trách: M2 (Chu Thành Tân).
- K-Means: K = 5, cách `weighted`, n_init = 50 (mục 3.2–3.3).
- DBSCAN: cấu hình chính B (1% số chuyến, eps 2000 m) và cấu hình phụ C (0,5%, eps 1000 m) (mục 4.4).

Nội dung:
1. Bảng chéo cụm K-Means × cụm DBSCAN theo số chuyến, cùng ARI.
2. Các vùng hai thuật toán khác nhau, có bản đồ phóng to.
3. Điểm nhiễu của DBSCAN là ngoại lệ thật hay do eps quá nhỏ.

Hai mô hình được **fit lại** từ config (kết quả tất định, chạy nhanh), nên notebook chạy được trên máy khác mà không cần file nhãn. Nếu có file nhãn của lần chạy chính thức (mục 3.3, 4.4) thì đối chiếu để chắc kết quả trùng.

Dữ liệu: **mẫu pilot tạm của M2**. Đổi `DATA_KEY = "model_input"` khi có dữ liệu của M1.

In [ ]:
# ===== O THIET LAP (chep nguyen o nay vao dau moi notebook) =====
# Colab: clone repo, cai thu vien, them goc repo vao sys.path.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Du lieu that tren Colab: bo comment 2 dong duoi khi nhom da chot thu muc Drive
# from src.colab import link_drive_data
# link_drive_data("KhaiPhaDuLieu")

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## Tham số và fit lại hai mô hình

In [ ]:
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score
from src.config import load_config, resolve_path
from src.evaluation import read_run_log
from src.evaluation.hotspots import relabel_by_size
from src.models.kmeans import aggregate_unique, fit_weighted as km_fit
from src.models.dbscan import fit_weighted as db_fit, min_samples_from_frac

cfg = load_config()
DATA_KEY = "pilot_m2"
MANIFEST_KEY = "pilot_m2_manifest" if DATA_KEY == "pilot_m2" else "manifest"
SLICES = list(cfg["time_slices"]); KM = cfg["kmeans"]; DB = cfg["dbscan"]; SEED = cfg["project"]["seed"]
df = pd.read_parquet(resolve_path(cfg, DATA_KEY))
SAMPLE_ID = json.loads(resolve_path(cfg, MANIFEST_KEY).read_text(encoding="utf-8"))["sample_id"]
N_INIT = 50   # nhu muc 3.2-3.3

res = {}
for s in SLICES:
    part = df[df.time_slice == s].reset_index(drop=True)
    X = part[["x_m", "y_m"]].to_numpy("float64")
    km_lab = relabel_by_size(km_fit(X, KM["k_final"], init=KM["init"], n_init=N_INIT, max_iter=KM["max_iter"], seed=SEED)[0])
    out = {"part": part, "X": X, "kmeans": km_lab}
    for name, conf in (("dbB", DB["final"]), ("dbC", DB["secondary"])):
        ms = min_samples_from_frac(len(X), conf["min_samples_frac"])
        lab, db = db_fit(X, float(conf["eps_m"]), ms)
        out[name] = relabel_by_size(lab); out[name + "_model"] = db; out[name + "_ms"] = ms
    res[s] = out

# Doi chieu voi nhan da luu cua lan chay chinh thuc (neu co)
log = read_run_log(resolve_path(cfg, "run_log"))
log = log[log.sample_id == SAMPLE_ID]
def saved(algo, s, role=None):
    d = log[(log.algorithm == algo) & (log.time_slice == s)]
    if role: d = d[d.note.str.contains(role, regex=False)]
    if d.empty: return None
    f = resolve_path(cfg, "labels_dir") / f"{d.run_id.iloc[-1]}.parquet"
    return pd.read_parquet(f).label.to_numpy() if f.exists() else None
for s in SLICES:
    for algo, key, role in (("kmeans", "kmeans", None), ("dbscan", "dbB", "chinh (B)"), ("dbscan", "dbC", "phu (C)")):
        lab = saved(algo, s, role)
        status = "khong co file nhan" if lab is None else ("TRUNG" if np.array_equal(lab, res[s][key]) else "KHAC")
        print(f"{s:11s} {key:6s}: {status}")

## 1. Bảng chéo K-Means × DBSCAN (B)

Mỗi ô là số chuyến. Cột `-1` là nhiễu của DBSCAN. ARI tính theo 2 cách: (a) trên mọi chuyến, coi nhiễu là một nhãn riêng; (b) chỉ trên các chuyến DBSCAN có gán cụm.

In [ ]:
rows = []
for s in SLICES:
    r = res[s]; km, dbB, dbC = r["kmeans"], r["dbB"], r["dbC"]
    m = dbB != -1
    rows.append({"time_slice": s, "n": len(km),
                 "ari_all_B": adjusted_rand_score(km, dbB), "ari_clustered_B": adjusted_rand_score(km[m], dbB[m]),
                 "ari_all_C": adjusted_rand_score(km, dbC), "ari_B_vs_C": adjusted_rand_score(dbB, dbC)})
    ct = pd.crosstab(pd.Series(km, name="kmeans"), pd.Series(dbB, name="dbscan_B"))
    print(f"--- {s} (so chuyen) ---"); print(ct.to_string()); print()
ari = pd.DataFrame(rows)
ari.round(3)

In [ ]:
# Ty le moi cum K-Means bi DBSCAN B gan nhieu, va cum DBSCAN B ma no roi vao nhieu nhat
rows = []
for s in SLICES:
    r = res[s]
    for c in range(KM["k_final"]):
        sel = r["kmeans"] == c
        db_in = r["dbB"][sel]
        main = pd.Series(db_in[db_in != -1]).value_counts()
        rows.append({"time_slice": s, "kmeans_cluster": c, "n_trips": int(sel.sum()),
                     "share_noise_in_B_pct": round((db_in == -1).mean() * 100, 1),
                     "main_dbB_cluster": int(main.index[0]) if len(main) else -1,
                     "main_dbB_share_pct": round(main.iloc[0] / sel.sum() * 100, 1) if len(main) else 0.0,
                     "top_community_area": r["part"].loc[sel, "pickup_community_area"].mode().iloc[0]})
km_vs_db = pd.DataFrame(rows)
km_vs_db

## 2. Điểm nhiễu của DBSCAN B: ngoại lệ thật hay do eps quá nhỏ?

Với mỗi toạ độ bị gán nhiễu ở B:
- `dist_to_core_m`: khoảng cách tới toạ độ core gần nhất. Nếu nhỏ hơn eps nhưng vẫn là nhiễu thì không thể xảy ra; nếu chỉ lớn hơn eps một chút thì đó là "suýt được nối", tức nhiễu do eps.
- Số chuyến của toạ độ đó: toạ độ ít chuyến và ở xa thì là vùng thưa thật.
- Ở cấu hình phụ C (eps nhỏ hơn nhưng `min_samples` cũng nhỏ hơn), toạ độ đó có được gán cụm không.
- Tỷ lệ cờ chất lượng (`flag_*` do bước làm sạch gắn) của chuyến nhiễu so với chuyến có cụm: nếu không cao hơn thì nhiễu **không phải** do bản ghi lỗi.

In [ ]:
from sklearn.neighbors import NearestNeighbors
rows, flag_rows = [], []
for s in SLICES:
    r = res[s]; X = r["X"]; part = r["part"]
    Xu, w, inv = aggregate_unique(X)
    db = r["dbB_model"]; lu = db.labels_
    core = Xu[db.core_sample_indices_]
    noise_u = np.where(lu == -1)[0]
    d = NearestNeighbors(n_neighbors=1).fit(core).kneighbors(Xu[noise_u])[0][:, 0]
    luC = r["dbC_model"].labels_
    for j, u in enumerate(noise_u):
        rows.append({"time_slice": s, "x_m": Xu[u, 0], "y_m": Xu[u, 1], "n_trips": int(w[u]),
                     "dist_to_core_m": float(d[j]), "clustered_in_C": bool(luC[u] != -1)})
    noise_trip = r["dbB"] == -1
    for f in [c for c in part.columns if c.startswith("flag_")]:
        flag_rows.append({"time_slice": s, "flag": f,
                          "noise_pct": round(part.loc[noise_trip, f].mean() * 100, 2),
                          "clustered_pct": round(part.loc[~noise_trip, f].mean() * 100, 2)})
noise = pd.DataFrame(rows)
eps_B = DB["final"]["eps_m"]
summ = noise.groupby("time_slice").agg(
    noise_coords=("n_trips", "size"), noise_trips=("n_trips", "sum"),
    trips_per_coord_median=("n_trips", "median"), trips_per_coord_max=("n_trips", "max"),
    dist_median_m=("dist_to_core_m", "median"),
    near_eps_coords=("dist_to_core_m", lambda v: int(((v > eps_B) & (v <= 1.25 * eps_B)).sum())),
    far_coords=("dist_to_core_m", lambda v: int((v > 2 * eps_B).sum())),
    clustered_in_C_coords=("clustered_in_C", "sum"))
clustered_med = {s: float(np.median(aggregate_unique(res[s]["X"])[1][res[s]["dbB_model"].labels_ != -1])) for s in SLICES}
summ["trips_per_clustered_coord_median"] = pd.Series(clustered_med)
summ["clustered_in_C_trips_pct"] = noise[noise.clustered_in_C].groupby("time_slice").n_trips.sum() / summ.noise_trips * 100
print("eps B =", eps_B, "m; 'near_eps' = khoang cach toi core trong (eps, 1.25 eps]; 'far' = > 2 eps")
summ.round(1)

In [ ]:
print("Ty le co chat luong (%): chuyen nhieu (B) so voi chuyen co cum")
pd.DataFrame(flag_rows).pivot(index="flag", columns="time_slice", values=["noise_pct", "clustered_pct"])

## 3. Bản đồ phóng to các vùng hai thuật toán khác nhau (WEEKDAY_PM)

Cùng khung nhìn cho K-Means và DBSCAN B. Mỗi chấm là một toạ độ, kích thước theo số chuyến; dấu x xám là nhiễu của DBSCAN. Toạ độ x/y tính bằng km (EPSG:26916).

In [ ]:
REGIONS = {"R1 Phia nam": (440, 457, 4610, 4632), "R2 O'Hare va tay bac": (420, 442, 4638, 4653),
           "R3 Phia bac doc ho": (440, 452, 4636, 4653)}
S = "WEEKDAY_PM"
r = res[S]; Xu, w, inv = aggregate_unique(r["X"])
lab_u = {"K-Means K=5": r["kmeans"][np.unique(inv, return_index=True)[1]],
         "DBSCAN B (1%, 2000 m)": r["dbB"][np.unique(inv, return_index=True)[1]]}
fig, axes = plt.subplots(len(REGIONS), 2, figsize=(10, 4.6 * len(REGIONS)))
for i, (rname, (x0, x1, y0, y1)) in enumerate(REGIONS.items()):
    inside = (Xu[:, 0] / 1000 >= x0) & (Xu[:, 0] / 1000 <= x1) & (Xu[:, 1] / 1000 >= y0) & (Xu[:, 1] / 1000 <= y1)
    for j, (aname, lu) in enumerate(lab_u.items()):
        ax = axes[i, j]
        nz = lu == -1
        ax.scatter(Xu[nz, 0] / 1000, Xu[nz, 1] / 1000, marker="x", s=15 + 400 * w[nz] / w.max(), c="#777777", linewidths=1)
        ax.scatter(Xu[~nz, 0] / 1000, Xu[~nz, 1] / 1000, s=8 + 800 * w[~nz] / w.max(), c=lu[~nz] % 10, cmap="tab10",
                   vmin=0, vmax=9, alpha=.8, edgecolors="k", linewidths=.3)
        trips_in = int(w[inside].sum()); noise_in = int(w[inside & nz].sum())
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal"); ax.grid(alpha=.3)
        ax.set_title(f"{rname} - {aname}" + chr(10) + f"{trips_in:,} chuyen trong khung, nhieu {noise_in:,}", fontsize=9)
fig.suptitle(f"Vung bat dong K-Means vs DBSCAN - {S} - {SAMPLE_ID}")
fig.tight_layout()
fig.savefig(resolve_path(cfg, "figures_dir") / f"m2_compare_regions_{SAMPLE_ID}.png", dpi=120)

In [ ]:
# So chuyen theo vung va theo nhan cua tung thuat toan (ca 3 lat cat)
rows = []
for s in SLICES:
    r = res[s]; X = r["X"] / 1000
    for rname, (x0, x1, y0, y1) in REGIONS.items():
        sel = (X[:, 0] >= x0) & (X[:, 0] <= x1) & (X[:, 1] >= y0) & (X[:, 1] <= y1)
        rows.append({"time_slice": s, "region": rname, "n_trips": int(sel.sum()),
                     "kmeans_clusters": sorted(set(r["kmeans"][sel].tolist())),
                     "dbB_noise_pct": round((r["dbB"][sel] == -1).mean() * 100, 1),
                     "dbB_clusters": sorted(set(r["dbB"][sel].tolist()) - {-1})})
regions = pd.DataFrame(rows)
cmp_path = resolve_path(cfg, "run_log").parent / f"m2_compare_{SAMPLE_ID}.csv"
pd.concat([ari.assign(table="ari"), km_vs_db.assign(table="kmeans_vs_dbscanB"),
           summ.reset_index().assign(table="noise_summary"), regions.astype({"kmeans_clusters": str, "dbB_clusters": str}).assign(table="regions")],
          ignore_index=True).to_csv(cmp_path, index=False)
print("Da luu:", cmp_path.relative_to(cfg["_root"]))
regions